In [0]:
df = spark.table("indian_roads.raw.accidents")
print(f"rows:{dp.count()}")

In [0]:
df = spark.table ("indian_roads.processed.accidents")
print(f"rows:{dp.count()}")

In [0]:
df = spark.table("indian_roads.raw.accidents")
print (f"rows:{df.count()}")
print (f"calumn:{df.count()}")
df.printSchema()

In [0]:
df = spark.table("indian_roads.raw.accidents")
print (f"rows:{df.count()}")
print (f"calumn:{len(df.columns)}")
df.printSchema()
display (df.limit(10))
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
df = spark.table("indian_roads.raw.accidents")
print (f"rows:{df.count()}")
print (f"calumn:{df.count()}")
df.printSchema()
display(df.limit(10))
from pyspark.sql.functions import col, sum as spark_sum
missing = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
missing = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])

In [0]:
display(missing)

In [0]:
display(
    df.groupBy("accident_severity")
      .count()
      .orderBy("count", ascending=  False)
)

In [0]:
display(
    df.groupBy("cause")
      .count()
      .orderBy("count", ascending=False)
)

In [0]:
display(
    df.groupBy("road_type")
      .count()
      .orderBy("count", ascending=False)
)

In [0]:
display(
    df.groupBy("weather")
      .count()
      .orderBy("count", ascending=False)
)

In [0]:
display(
    df.groupBy("city")
      .count()
      .orderBy("count", ascending=False)
      .limit(10)
)

In [0]:
display(
    df.groupBy("hour")
      .count()
      .orderBy("hour")        
)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ── 1. Convert Spark → Pandas ──────────────────────────────────────────
pdf = df.toPandas()
# df is our Spark DataFrame from earlier
# toPandas() pulls it into memory so we can use matplotlib/seaborn

# ── 2. Basic Dataset Info ──────────────────────────────────────────────
print("=" * 55)
print("         INDIAN ROADS ACCIDENT DATASET - SUMMARY")
print("=" * 55)
print(f"Total accidents recorded : {len(pdf):,}")
print(f"Number of columns        : {len(pdf.columns)}")
print(f"Date range               : {pdf['date'].min()} → {pdf['date'].max()}")
print(f"Cities covered           : {pdf['city'].nunique()}")
print(f"States covered           : {pdf['state'].nunique()}")

# ── 3. Severity Breakdown ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ACCIDENT SEVERITY BREAKDOWN")
print("=" * 55)
severity_counts = pdf['accident_severity'].value_counts()
severity_pct    = pdf['accident_severity'].value_counts(normalize=True) * 100

for sev in severity_counts.index:
    bar = "█" * int(severity_pct[sev] / 2)   # simple text bar
    print(f"  {sev:<10} {severity_counts[sev]:>6,}  ({severity_pct[sev]:.1f}%)  {bar}")
# value_counts()              → counts each unique value
# value_counts(normalize=True)→ gives proportions (0 to 1)
# * 100                       → converts to percentage
# f"{sev:<10}"                → left-aligns the text in 10 characters
# f"{count:>6,}"              → right-aligns number with comma formatting

# ── 4. Most Common Accident Causes ────────────────────────────────────
print("\n" + "=" * 55)
print("  TOP CAUSES OF ACCIDENTS")
print("=" * 55)
cause_counts = pdf['cause'].value_counts()
cause_pct    = pdf['cause'].value_counts(normalize=True) * 100

for cause in cause_counts.index:
    bar = "█" * int(cause_pct[cause] / 2)
    print(f"  {cause:<20} {cause_counts[cause]:>6,}  ({cause_pct[cause]:.1f}%)  {bar}")

# ── 5. Which Cause Leads to Most FATAL Accidents ──────────────────────
print("\n" + "=" * 55)
print("  WHICH CAUSE LEADS TO MOST FATAL ACCIDENTS")
print("=" * 55)
fatal_by_cause = (
    pdf[pdf['accident_severity'] == 'fatal']   # filter only fatal rows
       ['cause']                                # pick the cause column
       .value_counts()                          # count each cause
)
fatal_pct_by_cause = fatal_by_cause / fatal_by_cause.sum() * 100
# pdf[pdf['accident_severity'] == 'fatal'] 
#   → filters the DataFrame to keep only rows where severity is fatal
#   → think of it like: "give me all rows WHERE severity = fatal"

for cause in fatal_by_cause.index:
    bar = "█" * int(fatal_pct_by_cause[cause] / 2)
    print(f"  {cause:<20} {fatal_by_cause[cause]:>5,} fatal  ({fatal_pct_by_cause[cause]:.1f}%)  {bar}")

# ── 6. Fatal Rate per Cause (most dangerous cause) ────────────────────
print("\n" + "=" * 55)
print("  FATAL RATE PER CAUSE  (% of accidents that were fatal)")
print("=" * 55)
fatal_rate = (
    pdf.groupby('cause')['accident_severity']   # group by cause, look at severity
       .apply(lambda x: (x == 'fatal').sum() / len(x) * 100)  # % that are fatal
       .sort_values(ascending=False)
)
# groupby('cause')  → splits data into groups, one per cause
# apply(lambda x: ) → runs a custom function on each group
# lambda x:         → a mini one-line function; x is the group
# (x == 'fatal').sum() / len(x) * 100 → fa
for cause, rate in fatal_rate.items():
    bar = "█" * int(rate / 2)
    print(f"  {cause:<20} {rate:.1f}% fatal  {bar}")

# ── 7. Weather vs Severity ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("  FATAL RATE BY WEATHER CONDITION")
print("=" * 55)
fatal_rate_weather = (
    pdf.groupby('weather')['accident_severity']
       .apply(lambda x: (x == 'fatal').sum() / len(x) * 100)
       .sort_values(ascending=False)
)
for weather, rate in fatal_rate_weather.items():
    bar = "█" * int(rate / 2)
    print(f"  {weather:<15} {rate:.1f}% fatal  {bar}")

# ── 8. Road Type vs Severity ───────────────────────────────────────────
print("\n" + "=" * 55)
print("  FATAL RATE BY ROAD TYPE")
print("=" * 55)
fatal_rate_road = (
    pdf.groupby('road_type')['accident_severity']
       .apply(lambda x: (x == 'fatal').sum() / len(x) * 100)
       .sort_values(ascending=False)
)
for road, rate in fatal_rate_road.items():
    bar = "█" * int(rate / 2)
    print(f"  {road:<15} {rate:.1f}% fatal  {bar}")

# ── 9. Peak Hour vs Off Peak ───────────────────────────────────────────
print("\n" + "=" * 55)
print("  PEAK HOUR vs OFF-PEAK FATAL RATE")
print("=" * 55)
peak_fatal = (
    pdf.groupby('is_peak_hour')['accident_severity']
       .apply(lambda x: (x == 'fatal').sum() / len(x) * 100)
)
# is_peak_hour is 1 (yes) or 0 (no) — an integer flag column
labels = {0: "Off-peak", 1: "Peak hour"}
for key, rate in peak_fatal.items():
    bar = "█" * int(rate / 2)
    print(f"  {labels[key]:<15} {rate:.1f}% fatal  {bar}")

# ── 10. Final Verdict ──────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  MOST DANGEROUS COMBINATION")
print("=" * 55)
combo = (
    pdf.groupby(['cause', 'weather', 'road_type'])['accident_severity']
       .apply(lambda x: (x == 'fatal').sum() / len(x) * 100)
       .reset_index(name='fatal_pct')         # rename the result column
       .sort_values('fatal_pct', ascending=False)
       .head(5)                               # top 5 deadliest combos
)
# groupby with multiple columns → groups by every unique combination
# reset_index(name='fatal_pct') → turns the result back into a clean DataFrame

print("\n  Top 5 deadliest cause + weather + road combinations:\n")
for _, row in combo.iterrows():
    # iterrows() loops through a DataFrame row by row
    # _ is the index (we don't need it), row has the values
    print(f"  {row['cause']:<20} + {row['weather']:<8}"
          f" + {row['road_type']:<10}  →  {row['fatal_pct']:.1f}% fatal")

print("\n" + "=" * 55)
print("  END OF SUMMARY")
print("=" * 55)

In [0]:
# Convert Spark DataFrame to Pandas
pdf = df.toPandas()

# Quick check
print(pdf.shape)
print(pdf['accident_severity'].value_counts())

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count accidents by weather and severity
weather_severity = pdf.groupby(['weather', 'accident_severity']).size().unstack()

# Plot
weather_severity.plot(kind='bar', figsize=(10, 6), colormap='RdYlGn_r')
plt.title('Accident Severity by Weather Condition')
plt.xlabel('Weather')
plt.ylabel('Number of Accidents')
plt.xticks(rotation=0)
plt.legend(title='Severity')
plt.tight_layout()
plt.show()

In [0]:
cause_severity = pdf.groupby(['cause', 'accident_severity']).size().unstack()

cause_severity.plot(kind='bar', figsize=(12, 6), colormap='RdYlGn_r')
plt.title('Accident Severity by Cause')
plt.xlabel('Cause')
plt.ylabel('Number of Accidents')
plt.xticks(rotation=15)
plt.legend(title='Severity')
plt.tight_layout()
plt.show()

In [0]:
road_severity = pdf.groupby(['road_type', 'accident_severity']).size().unstack()

road_severity.plot(kind='bar', figsize=(10, 6), colormap='RdYlGn_r')
plt.title('Accident Severity by Road Type')
plt.xlabel('Road Type')
plt.ylabel('Number of Accidents')
plt.xticks(rotation=0)
plt.legend(title='Severity')
plt.tight_layout()
plt.show()